# Create the main life-cycle inventory database in your brightway project

In [ ]:
%pip install brightway2
%pip install premise
%pip install mescal
%pip install carculator_truck

In [1]:
import bw2data as bd
import bw2io as bi
from premise import *
from mescal import *
from carculator_truck import *
from datetime import datetime

In [3]:
ei_version = '3.12' # version of the ecoinvent database. Careful: carculator_truck v0.5.0 is compatible with ecoinvent 3.10 only
ei_db_name = f"ecoinvent-{ei_version}-cutoff" # name of the ecoinvent database in the brightway project
bd.projects.set_current(f'ecoinvent{ei_version}') # set the current brightway project
biosphere_name= 'biosphere3' # name of the biosphere database in the brightway project

### Premise parameters ###
model = 'image' # IAM used to create the database
pathway = 'SSP2-M' # SSP-RCP scenario used to create the database
years = [2023] # years of the database

# Or, define a complete list of scenarios
use_scenarios_list = False
scenarios = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [3]:
if ei_version == '3.10.1':
    ei_version_premise = '3.10'
else:
    ei_version_premise = ei_version

## Import ecoinvent

In [ ]:
if f'ecoinvent-{ei_version}-cutoff' in bd.databases:
    print(f'ecoinvent {ei_version} is already present in the project')
else:
    bi.import_ecoinvent_release(
        version=ei_version,
        system_model='cutoff', # can be cutoff / apos / consequential / EN15804
        username='JohnDoe',
        password='1234',
        biosphere_name=biosphere_name,
    )

## Import premise databases

In [4]:
# Clear cache is encouraged if updating premise or if encountering issues with inventories
clear_cache()

Cache folder cleared!


In [5]:
# Initialize the premise database
ndb = NewDatabase(
    scenarios=scenarios if use_scenarios_list else [{"model": model, "pathway": pathway, "year": year} for year in years],
    source_db=ei_db_name,
    source_version=ei_version_premise,
    key='xxx', # ask a key to Romain Sacchi (romain.sacchi@psi.ch)
    biosphere_name=biosphere_name,
)

premise v.(2, 4, 9)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+-------------

100%|██████████| 26533/26533 [00:03<00:00, 7382.42it/s] 


Adding exchange data to activities


100%|██████████| 892943/892943 [00:34<00:00, 25895.95it/s]


Filling out exchange data


100%|██████████| 26533/26533 [00:02<00:00, 10160.00it/s]


Set missing location of datasets to global scope.
Set missing location of production exchanges to scope of dataset.
Correct missing location of technosphere exchanges.
Correct missing flow categories for biosphere exchanges
Remove empty exchanges.
Remove uncertainty data.
- Extracting inventories
Cannot find cached inventories. Will create them now for next time...
Importing default inventories...

Importing C:\Users\matth\PycharmProjects\EnergyScope-Quebec\.venv\Lib\site-packages\premise\data\additional_inventories\lci-Carma-CCS.xlsx
Extracted 1 worksheets in 0.12 seconds
Migration route: 3.5 → 3.6 → 3.7 → 3.8 → 3.9 → 3.10 → 3.11 → 3.12
Applying forward migration 3.5 -> 3.6
Applying forward migration 3.6 -> 3.7
Applying forward migration 3.7 -> 3.8
Applying forward migration 3.8 -> 3.9
Applying forward migration 3.9 -> 3.10
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing C:\Users\matth\PycharmProjects\EnergyScope-Quebec\.venv\Lib\site-packages

In [ ]:
# Update the database with IAM data
ndb.update()

In [ ]:
databases_names = (
    [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for model, pathway, year in [(s['model'], s['pathway'], s['year']) for s in scenarios]] if use_scenarios_list
    else [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for year in years]
)

In [6]:
# Write the database in your brightway project
ndb.write_db_to_brightway(name=databases_names)

Write new database(s) to Brightway.
Running all checks...
Minor anomalies found: check the change report.


Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:01:09


Title: Writing activities to SQLite3 database:
  Started: 09/08/2026 10:03:40
  Finished: 09/08/2026 10:04:49
  Total time elapsed: 00:01:09
  CPU %: 90.00
  Memory %: 12.75
Created database: ecoinvent_cutoff_3.12_2023
Brightway database written: ecoinvent_cutoff_3.12_2023
Generate scenario report.
Report saved under C:\Users\matth\PycharmProjects\EnergyScope-Quebec\projects\lca\01_Setup\export\scenario_report.
Generate change report.
Report saved under C:\Users\matth\PycharmProjects\EnergyScope-Quebec\projects\lca\01_Setup/export/change reports/.


In [ ]:
# Or the two at the same time
ndb.update_and_write(name=databases_names)

## Import carculator_truck databases

In [ ]:
def create_truck_database(cycle, year):
    tip = TruckInputParameters()
    tip.static()
    dcts, array = fill_xarray_from_input_parameters(tip)
    array = array.interp(year=[year],  kwargs={'fill_value': 'extrapolate'})
    tm = TruckModel(array, cycle=cycle)
    tm.set_all()
    ic = InventoryTruck(tm)
    
    i = ic.export_lci(
        software="brightway2",
        ecoinvent_version=ei_version_premise,
        format="bw2io",
        filename=cycle.lower(),
    )
    
    i.apply_strategies()

    i.match_database(fields=["name", "unit", "location"])
    i.match_database(ei_db_name,  fields=["reference product", "name", "unit", "location"])
    i.match_database(biosphere_name,  fields=["name", "unit", "categories"])

    i.statistics()
    
    i.drop_unlinked(i_am_reckless=True)  # remove noise elementary flows (not characterized)

    if cycle.lower() + f"_truck_{datetime.now().strftime('%Y%m%d')}_{year}" in bd.databases:
        del bd.databases[cycle.lower() + f"_truck_{datetime.now().strftime('%Y%m%d')}_{year}"]
    
    i.write_database()

In [ ]:
for year in years:
    create_truck_database(cycle='Urban delivery', year=year) # 150 km
    create_truck_database(cycle='Regional delivery', year=year) # 400 km
    create_truck_database(cycle='Long haul', year=year) # 800 km

## Minor changes in naming to the carculator_truck databases

In [ ]:
def rename_truck_activities(cycle, year):
    db = bd.Database(f"{cycle.lower()}_truck_{datetime.now().strftime('%Y%m%d')}_{year}")
    db_list = [a for a in db]
    for act in db_list:
        if act.as_dict()['name'].startswith('transport, truck') or act.as_dict()['name'].startswith('truck,'):
            act.as_dict()['name'] += f', {cycle.lower()}'
        act.save()

In [ ]:
for year in years:
    rename_truck_activities(cycle='Regional delivery', year=year)
    rename_truck_activities(cycle='Urban delivery', year=year)
    rename_truck_activities(cycle='Long haul', year=year)

## Merge the premise and carculator_truck databases into a single database

In [ ]:
for year in years:
    main_db = Database(f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}')
    carculator_truck_urban_db = Database(f"urban delivery_truck_{datetime.now().strftime('%Y%m%d')}_{year}")
    carculator_truck_regional_db = Database(f"regional delivery_truck_{datetime.now().strftime('%Y%m%d')}_{year}")
    carculator_truck_long_db = Database(f"long haul_truck_{datetime.now().strftime('%Y%m%d')}_{year}")

    # concatenation of the database (the lists of dictionaries are simply added up)
    db = main_db + carculator_truck_urban_db + carculator_truck_regional_db + carculator_truck_long_db
    
    db.merge(
        main_ecoinvent_db_name=f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}',
        old_main_db_names=[f'ecoinvent-{ei_version}-cutoff'],
        new_db_name=f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}+truck_carculator',
        write=True,
        check_duplicates=True,
        based_on='name',
    )  # performs relinking of secondary databases towards the main database, as well as elimination of duplicates

## Regionalization of the database

To regionalize the database clone the [Regiopremise](https://github.com/matthieu-str/Regiopremise) repository and run the [demo.ipynb](https://github.com/matthieu-str/Regiopremise/blob/ei3.10/doc/demo.ipynb) notebook. 

## Archiving, Restoring, and Sharing Projects

In [4]:
bi.backup.backup_project_directory(
    project=f'ecoinvent{ei_version}',
)

Creating project backup archive - this could take a few minutes...


'ecoinvent3.12'

In [ ]:
bi.backup.restore_project_directory(
    fp='./export/backup/brightway2-project-ecoinvent3.10.1-backup.30-May-2026-05-02AM.tar.gz',
    project_name=f'ecoinvent{ei_version}',
    overwrite_existing=True,
)